# Airline Flight Delay Analysis
**Data Analyst Training Project** | Domain: Airline Flights & Operations (Domestic Flight Delays Across Indian Metro Cities)

- **Student:** Shivansh Singh (B.Tech / BCA, First Year • CSE-13)
- **Dataset:** `AIRLINE_FLIGHTS_DATA.CSV` / `airline_flights_data(1).csv` (255 raw rows → 245 clean rows after hygiene)


## 1. Setup & Data Loading
Import required libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and load the raw dataset.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Update path if your CSV is stored in a different directory
DATA_PATH = 'data/raw/AIRLINE_FLIGHTS_DATA.CSV'
df_raw = pd.read_csv(DATA_PATH)

print(f'Raw shape: {df_raw.shape}')
df_raw.head()


## 2. Data Inspection
Inspect column data types, missing values, summary statistics, and anomalies.


In [ ]:
print('--- Info & Dtypes ---')
print(df_raw.info())

print('\n--- Missing Values ---')
print(df_raw.isnull().sum())

print('\n--- Exact Duplicates ---')
print('Duplicate rows:', df_raw.duplicated().sum())

print('\n--- Numeric Summary ---')
df_raw.describe(include='all')


## 3. Data Hygiene & Cleaning
Key cleaning actions performed:
1. **Inconsistent City Names:** Standardize casing using `.str.strip().str.title()` (e.g., collapsing casing variants across metro cities).
2. **Negative / Invalid Flight Distances:** Replace invalid negative distances (e.g., down to `-450 km`) with the median flight distance (`1,193 km`).
3. **Missing Flight Dates & Exact Duplicates:** Parse `Flight_Date`, drop rows with missing dates, and drop exact duplicate records so that **245 clean rows** remain out of 255.


In [ ]:
df = df_raw.copy()

# 1. Standardize city names (title casing & whitespace cleanup)
for col in ['Origin_City', 'Destination_City']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

# 2. Parse Flight_Date and drop rows with missing dates
if 'Flight_Date' in df.columns:
    df['Flight_Date'] = pd.to_datetime(df['Flight_Date'], errors='coerce')
    df = df.dropna(subset=['Flight_Date'])

# 3. Drop exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

# 4. Impute invalid (<= 0 or negative) Flight_Distance_Km with median (1193 km)
if 'Flight_Distance_Km' in df.columns:
    valid_dist_median = df.loc[df['Flight_Distance_Km'] > 0, 'Flight_Distance_Km'].median()
    if pd.isna(valid_dist_median):
        valid_dist_median = 1193.0
    df.loc[df['Flight_Distance_Km'] <= 0, 'Flight_Distance_Km'] = valid_dist_median
    df['Flight_Distance_Km'] = df['Flight_Distance_Km'].fillna(valid_dist_median)

# Impute any remaining numeric nulls with median and categorical nulls with mode
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=['object', 'string']).columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print(f'Rows before cleaning: {len(df_raw)} | Rows after cleaning: {len(df)}')


## 4. Feature Engineering
Create the five derived analytical features:
- `Delay_Difference_Min`: `Arrival_Delay_Min - Departure_Delay_Min` (measures time gained or lost in air/turnaround)
- `Delay_Band`: `On Time` (≤ 0), `Minor` (1–15), `Mod` (16–45), `Major` (> 45)
- `Price_Per_Km`: `(Ticket_Price_INR / Flight_Distance_Km).round(2)`
- `Flight_Month`: `Flight_Date.dt.month_name()`
- `Route`: `Origin_City + ' → ' + Destination_City`


In [ ]:
# 1. Delay_Difference_Min
df['Delay_Difference_Min'] = df['Arrival_Delay_Min'] - df['Departure_Delay_Min']

# 2. Delay_Band
def categorize_delay(mins):
    if mins <= 0:
        return 'On Time'
    elif mins <= 15:
        return 'Minor'
    elif mins <= 45:
        return 'Mod'
    else:
        return 'Major'

df['Delay_Band'] = df['Arrival_Delay_Min'].apply(categorize_delay)

# 3. Price_Per_Km
df['Price_Per_Km'] = (df['Ticket_Price_INR'] / df['Flight_Distance_Km'].replace(0, np.nan)).round(2)

# 4. Flight_Month
df['Flight_Month'] = df['Flight_Date'].dt.month_name()

# 5. Route
df['Route'] = df['Origin_City'] + ' → ' + df['Destination_City']

df[['Flight_ID', 'Route', 'Flight_Month', 'Arrival_Delay_Min', 'Delay_Difference_Min', 'Delay_Band', 'Price_Per_Km']].head()


## 5. Exploratory Analysis & Visualizations (Q1 – Q4)
### Q1: Overall Flight Delay Pattern (`Arrival_Delay_Min`, `Delay_Band`)


In [ ]:
mean_arr_delay = df['Arrival_Delay_Min'].mean()
median_arr_delay = df['Arrival_Delay_Min'].median()
q25, q75 = df['Arrival_Delay_Min'].quantile([0.25, 0.75])
iqr_val = q75 - q25

print(f'Mean Arrival Delay: {mean_arr_delay:.1f} min')
print(f'Median Arrival Delay: {median_arr_delay:.1f} min (IQR: {q25:.0f}–{q75:.0f} min, span {iqr_val:.0f} min)')
print('\nDelay Band Counts:')
print(df['Delay_Band'].value_counts())

# Chart 1: Delay Band Distribution & Arrival Delay Distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
band_order = ['On Time', 'Minor', 'Mod', 'Major']
band_counts = df['Delay_Band'].value_counts().reindex(band_order, fill_value=0)
axes[0].bar(band_counts.index, band_counts.values, color=['#2e7d32', '#f9a825', '#ef6c00', '#c62828'])
axes[0].set_title('Q1a: Flight Count by Delay Band')
axes[0].set_xlabel('Delay Band')
axes[0].set_ylabel('Number of Flights')
for i, v in enumerate(band_counts.values):
    axes[0].text(i, v + 1.5, str(v), ha='center', fontweight='bold')

axes[1].hist(df['Arrival_Delay_Min'], bins=20, color='#1565c0', edgecolor='white', alpha=0.85)
axes[1].axvline(mean_arr_delay, color='#d84315', linestyle='--', linewidth=2, label=f'Mean ({mean_arr_delay:.1f}m)')
axes[1].axvline(median_arr_delay, color='#2e7d32', linestyle='-', linewidth=2, label=f'Median ({median_arr_delay:.1f}m)')
axes[1].set_title('Q1b: Distribution of Arrival Delay (Minutes)')
axes[1].set_xlabel('Arrival Delay (Min)')
axes[1].set_ylabel('Frequency')
axes[1].legend()
plt.tight_layout()
plt.show()


### Q2: Departure Delay, Weather & Arrival Delay (`Departure_Delay_Min`, `Weather_Impact_Score`)


In [ ]:
corr_cols = ['Departure_Delay_Min', 'Weather_Impact_Score', 'Flight_Distance_Km', 'Arrival_Delay_Min']
available_corr = [c for c in corr_cols if c in df.columns]
corr_matrix = df[available_corr].corr()
print('Correlation with Arrival_Delay_Min:')
print(corr_matrix['Arrival_Delay_Min'].sort_values(ascending=False))

# Chart 2: Departure Delay vs Arrival Delay & Correlation Heatmap
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.regplot(data=df, x='Departure_Delay_Min', y='Arrival_Delay_Min', ax=axes[0],
            scatter_kws={'alpha': 0.6, 'color': '#1565c0'}, line_kws={'color': '#d84315'})
axes[0].set_title('Q2a: Departure Delay vs. Arrival Delay')
axes[0].set_xlabel('Departure Delay (Min)')
axes[0].set_ylabel('Arrival Delay (Min)')

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1])
axes[1].set_title('Q2b: Feature Correlation Matrix')
plt.tight_layout()
plt.show()


### Q3: Delays & Ticket Prices by Airline Type and Cabin Class


In [ ]:
by_airline = df.groupby('Airline_Type', as_index=False).agg(
    Flights=('Arrival_Delay_Min', 'count'),
    Mean_Arrival_Delay=('Arrival_Delay_Min', 'mean'),
    Mean_Ticket_Price_INR=('Ticket_Price_INR', 'mean'),
    Mean_Price_Per_Km=('Price_Per_Km', 'mean')
).sort_values('Mean_Arrival_Delay', ascending=False)

by_cabin = df.groupby('Cabin_Class', as_index=False).agg(
    Flights=('Arrival_Delay_Min', 'count'),
    Mean_Arrival_Delay=('Arrival_Delay_Min', 'mean'),
    Mean_Ticket_Price_INR=('Ticket_Price_INR', 'mean'),
    Mean_Price_Per_Km=('Price_Per_Km', 'mean')
).sort_values('Mean_Arrival_Delay', ascending=False)

print('--- By Airline Type ---')
print(by_airline.to_string(index=False))
print('\n--- By Cabin Class ---')
print(by_cabin.to_string(index=False))

# Chart 3: Airline Type & Cabin Class Delay + Fare Comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].barh(by_airline['Airline_Type'][::-1], by_airline['Mean_Arrival_Delay'][::-1], color='#ef6c00')
axes[0].set_title('Q3a: Mean Arrival Delay by Airline Type (Min)')
axes[0].set_xlabel('Mean Arrival Delay (Min)')

axes[1].barh(by_cabin['Cabin_Class'][::-1], by_cabin['Mean_Ticket_Price_INR'][::-1], color='#0277bd')
axes[1].set_title('Q3b: Mean Ticket Price by Cabin Class (INR)')
axes[1].set_xlabel('Mean Ticket Price (INR)')
plt.tight_layout()
plt.show()


### Q4: Routes & Flights with Largest Delays (`Route`, `Flight_ID`, `Arrival_Delay_Min`)


In [ ]:
route_summary = df.groupby('Route', as_index=False).agg(
    Sample_Size_n=('Arrival_Delay_Min', 'count'),
    Mean_Arrival_Delay=('Arrival_Delay_Min', 'mean'),
    Max_Arrival_Delay=('Arrival_Delay_Min', 'max')
).sort_values('Mean_Arrival_Delay', ascending=False)

print('--- Top Routes by Mean Arrival Delay (Check Sample Size n!) ---')
print(route_summary.head(10).to_string(index=False))

print('\n--- Lowest Routes by Mean Arrival Delay ---')
print(route_summary.tail(5).to_string(index=False))

top_flights = df.nlargest(10, 'Arrival_Delay_Min')[['Flight_ID', 'Route', 'Airline_Type', 'Cabin_Class', 'Arrival_Delay_Min']]
print('\n--- Top 10 Individual Flight Delays ---')
print(top_flights.to_string(index=False))

# Chart 4 & 5: Route Mean Delay & Top Individual Flights
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_routes_plot = route_summary.head(8).sort_values('Mean_Arrival_Delay', ascending=True)
bars = axes[0].barh(top_routes_plot['Route'], top_routes_plot['Mean_Arrival_Delay'], color='#3949ab')
axes[0].set_title('Q4a: Top Routes by Mean Arrival Delay (with n)')
axes[0].set_xlabel('Mean Arrival Delay (Min)')
for bar, (_, row) in zip(bars, top_routes_plot.iterrows()):
    axes[0].text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2,
                 f"{row['Mean_Arrival_Delay']:.1f}m (n={int(row['Sample_Size_n'])})",
                 va='center', fontsize=8.5)

top_flights_plot = top_flights.head(8).iloc[::-1]
labels = top_flights_plot['Flight_ID'].astype(str) + ' (' + top_flights_plot['Route'] + ')'
axes[1].barh(labels, top_flights_plot['Arrival_Delay_Min'], color='#c62828')
axes[1].set_title('Q4b: Highest Individual Flight Delays (Min)')
axes[1].set_xlabel('Arrival Delay (Min)')
plt.tight_layout()
plt.show()


## 6. Synthesis & Recommendations
- **Audit High-Delay Routes:** Investigate turnaround and congestion bottlenecks on high-delay corridors (such as `Kolkata → Hyderabad` and `Delhi → Mumbai`) while verifying route sample sizes (`n`).
- **Strengthen Dispatch Buffers:** Add schedule/turnaround buffers for `Low_Cost` and `Regional` carriers where average delays exceed full-service carriers.
- **Integrate Weather & Pricing Tiers:** Account for `Weather_Impact_Score` in proactive delay alerts and evaluate cabin pricing vs. punctuality.
